In [2]:
!pip install -U google-genai

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [ ]:
import os
import json
import time
import pandas as pd
from google import genai





# Load CSV Files

In [6]:
BASE_PATH = "/content"  # change if your files are in another folder

exp1 = pd.read_csv(f"exp1_results_scaled.csv")
exp2 = pd.read_csv(f"exp2_results_scaled.csv")
exp3 = pd.read_csv(f"exp3_results_scaled.csv")
exp4 = pd.read_csv(f"exp4_results_scaled.csv")

# Check shapes
print("Exp1:", exp1.shape)
print("Exp2:", exp2.shape)
print("Exp3:", exp3.shape)
print("Exp4:", exp4.shape)

Exp1: (120, 5)
Exp2: (120, 6)
Exp3: (120, 6)
Exp4: (120, 6)


In [7]:
exp1.head()

,true_class,predicted_class,confidence,direct_description,image_idx
0,annual crop land,annual crop land,100.0,"The model likely predicted ""annual crop land"" ...",0
1,annual crop land,annual crop land,100.0,"The model likely predicted ""annual crop land"" ...",1
2,annual crop land,annual crop land,100.0,"The image shows a pattern of uniform, rectangu...",2
3,annual crop land,annual crop land,99.5,"The model likely predicted ""annual crop land"" ...",3
4,annual crop land,annual crop land,99.9,"The model likely predicted ""annual crop land"" ...",4


In [8]:
exp2.head()

,true_class,predicted_class,confidence,gradcam_region,xai_explanation,image_idx
0,annual crop land,annual crop land,100.0,"bottom-left region, strongly activated","The model likely predicted ""annual crop land"" ...",0
1,annual crop land,annual crop land,100.0,"bottom-right region, strongly activated","The model likely predicted ""annual crop land"" ...",1
2,annual crop land,annual crop land,100.0,"bottom-right region, strongly activated","The model likely predicted ""annual crop land"" ...",2
3,annual crop land,annual crop land,99.5,"bottom-left region, strongly activated","The model likely predicted ""annual crop land"" ...",3
4,annual crop land,annual crop land,99.9,"bottom-right region, strongly activated","The model likely predicted ""annual crop land"" ...",4


In [9]:
exp3.head()

,true_class,predicted_class,confidence,lime_description,lime_explanation,image_idx
0,annual crop land,annual crop land,100.0,"5 important segments in the upper-center, lowe...","The model likely predicted ""annual crop land"" ...",0
1,annual crop land,annual crop land,100.0,"5 important segments in the middle-center, mid...","The model likely predicted ""annual crop land"" ...",1
2,annual crop land,annual crop land,100.0,"5 important segments in the middle-left, lower...","The model likely identified the area as ""annua...",2
3,annual crop land,annual crop land,99.5,"5 important segments in the upper-right, lower...","The model likely predicted ""annual crop land"" ...",3
4,annual crop land,annual crop land,99.9,"5 important segments in the lower-center, midd...","The model likely predicted ""annual crop land"" ...",4


In [10]:
exp4.head()

,true_class,predicted_class,confidence,shap_description,shap_explanation,image_idx
0,annual crop land,annual crop land,100.0,"middle-right and upper-left regions, moderatel...","The model predicted ""annual crop land"" likely ...",0
1,annual crop land,annual crop land,100.0,"middle-center and middle-right regions, modera...","The model likely predicted ""annual crop land"" ...",1
2,annual crop land,annual crop land,100.0,"upper-left and middle-left regions, moderately...",The CNN model likely identified patterns in th...,2
3,annual crop land,annual crop land,99.5,"middle-left and upper-left regions, moderately...","The model likely predicted ""annual crop land"" ...",3
4,annual crop land,annual crop land,99.9,"lower-left and upper-center regions, moderatel...","The model likely predicted ""annual crop land"" ...",4


# Align Rows

must ensure:
Same image = same row across all experiments
Otherwise comparisons become wrong

In [11]:
# Sort all dataframes the same way
sort_cols = ["true_class", "image_idx"]

exp1 = exp1.sort_values(sort_cols).reset_index(drop=True)
exp2 = exp2.sort_values(sort_cols).reset_index(drop=True)
exp3 = exp3.sort_values(sort_cols).reset_index(drop=True)
exp4 = exp4.sort_values(sort_cols).reset_index(drop=True)

# Check alignment
for i in range(5):
    print(
        exp1.loc[i, "true_class"], 
        exp1.loc[i, "image_idx"],
        "|",
        exp2.loc[i, "image_idx"],
        exp3.loc[i, "image_idx"],
        exp4.loc[i, "image_idx"]
    )

annual crop land 0 | 0 0 0
annual crop land 1 | 1 1 1
annual crop land 2 | 2 2 2
annual crop land 3 | 3 3 3
annual crop land 4 | 4 4 4


# Create LLM Judge Prompt

In [25]:
def build_judge_prompt(predicted_class, baseline_explanation, xai_explanation, xai_method):
    """
    Pairwise comparison: Baseline vs XAI-informed explanation
    
    Args:
        predicted_class: The class predicted by the model
        baseline_explanation: Explanation from Exp1 (image + prediction only)
        xai_explanation: Explanation from Exp2/3/4 (image + prediction + XAI guidance)
        xai_method: "GradCAM", "LIME", or "SHAP"
    """
    return f"""You are evaluating two explanations for why a CNN model predicted a specific class for a satellite image.

The model predicted: "{predicted_class}"

Explanation A:
\"\"\"{baseline_explanation}\"\"\"

Explanation B:
\"\"\"{xai_explanation}\"\"\"

Evaluate both explanations on these criteria:

1. Specificity (0-5 points):
   - Does the explanation mention concrete visual features?
   - Examples: shapes (rectangular, circular), textures (rough, smooth), 
     patterns (grid-like, uniform), colors (green, brown), 
     spatial layout (clustered, linear, scattered)
   - Avoid vague terms like "typical" or "characteristic"
   
   Score 5: Multiple specific visual features clearly described
   Score 3: Some specific features mentioned
   Score 1: Only vague, general descriptions
   Score 0: No visual features mentioned

2. Usefulness (0-5 points):
   - Does the explanation help a human understand the model's reasoning?
   - Does it connect visual features to the predicted class?
   - Would a non-technical user (city planner, policy maker) find this explanation helpful?
   
   Score 5: Very clear connection between features and prediction
   Score 3: Some helpful information but not fully clear
   Score 1: Minimal help in understanding the reasoning
   Score 0: Confusing or unhelpful

Total Score = Specificity + Usefulness (0 to 10)

IMPORTANT: Return ONLY valid JSON. Do NOT use markdown formatting or code blocks.

{{
  "score_A": <number 0-10>,
  "score_B": <number 0-10>,
  "specificity_A": <number 0-5>,
  "specificity_B": <number 0-5>,
  "usefulness_A": <number 0-5>,
  "usefulness_B": <number 0-5>,
  "winner": "A" or "B" or "Tie",
  "reason": "<brief 1-sentence explanation of why winner was chosen>"
}}
"""

# LLM Judge Function (Core Execution)

In [ ]:
from openai import OpenAI
import json
import re
import time

API_KEY = "key"

client = OpenAI(api_key=API_KEY)

In [113]:
def judge_with_openai(prompt):

    while True:
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,   # 🔥 important for consistency
                max_tokens=300
            )

            text = response.choices[0].message.content.strip()

            # Extract JSON safely
            match = re.search(r"\{.*\}", text, re.DOTALL)

            if match:
                json_text = match.group(0)
                result = json.loads(json_text)
                return result
            else:
                print("No JSON found:", text)
                return {
                    "score_A": None,
                    "score_B": None,
                    "winner": "Error",
                    "reason": "No JSON found"
                }

        except Exception as e:
            error_str = str(e)

            # 🔥 Handle rate limits
            if "429" in error_str:
                print("⏳ Rate limit hit. Waiting 5 seconds...")
                time.sleep(5)
                continue

            print("Error:", e)
            return {
                "score_A": None,
                "score_B": None,
                "winner": "Error",
                "reason": "Parsing failed"
            }

# Exp1 vs Exp2

In [ ]:
# Pick first sample
i = 100

predicted_class = exp1.loc[i, "predicted_class"]

# Exp1 = baseline
exp1_text = exp1.loc[i, "direct_description"]
print("exp1_text:")
print(exp1_text)

# Exp2 = GradCAM (you can later switch to exp3 or exp4)
exp2_text = exp2.loc[i, "xai_explanation"]
print("\nexp2_text:")
print(exp2_text)

# Build prompt (no bias mention)
prompt = build_judge_prompt(
    predicted_class,
    exp1_text,
    exp2_text,
    "GradCAM"   # not shown to model, just your variable
)

# Call openai
result = judge_with_openai(prompt)

print("\nResult:")
print(result)

exp1_text:
The model likely predicted the class "river" due to the elongated, winding shape visible in the image, typical of river formations. The smooth, continuous texture and the dark color of this feature contrast with the surrounding areas, which aligns with how water bodies often appear in satellite images. Additionally, the linear pattern running parallel to this feature may resemble banks or shorelines, reinforcing the identification of a river.

exp2_text:
The model likely predicted the class "river" because in the bottom-right region, there is a distinct elongated, winding shape consistent with a river. This area shows a different texture compared to the surrounding land, suggesting the presence of water. The continuous, curved pattern follows the characteristic flow of a river, helping the model to identify it accurately.
No JSON found: 

Result:
{'score_A': None, 'score_B': None, 'winner': 'Error', 'reason': 'No JSON found'}


In [79]:
results_base_vs_gradcam = []

for i in range(len(exp1)):

  #  time.sleep(15)
    
    predicted_class = exp1.loc[i, "predicted_class"]

    exp1_text = exp1.loc[i, "direct_description"]
    exp2_text = exp2.loc[i, "xai_explanation"]

    prompt = build_judge_prompt(
        predicted_class,
        exp1_text,
        exp2_text,
        "GradCAM"
    )

    result = judge_with_openai(prompt)

    winner_raw = result.get("winner")

    if winner_raw == "A":
        winner_label = "base"
    elif winner_raw == "B":
        winner_label = "GradCAM"
    elif winner_raw == "Tie":
        winner_label = "Tie"
    else:
        winner_label = "Error"

    results_base_vs_gradcam.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "score_base": result.get("score_A"),
        "score_gradcam": result.get("score_B"),
        "winner": winner_label,
        "specificity_base": result.get("specificity_A"),
        "specificity_gradcam": result.get("specificity_B"),
        "usefulness_base": result.get("usefulness_A"),
        "usefulness_gradcam": result.get("usefulness_B"),
        "reason": result.get("reason")
    })

    # IMPORTANT: avoid rate limit
    #time.sleep(12)

    # Progress update
    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [80]:
df_exp12 = pd.DataFrame(results_base_vs_gradcam)

df_exp12.head()

,index,true_class,predicted_class,score_base,score_gradcam,winner,specificity_base,specificity_gradcam,usefulness_base,usefulness_gradcam,reason
0,0,annual crop land,annual crop land,9,8,base,5,4,4,4,Explanation A provides a slightly more detaile...
1,1,annual crop land,annual crop land,8,7,base,4,3,4,4,Explanation A provides more specific visual fe...
2,2,annual crop land,annual crop land,9,7,base,5,3,4,4,Explanation A provides multiple concrete visua...
3,3,annual crop land,annual crop land,8,9,GradCAM,4,5,4,4,Explanation B provides more specific visual de...
4,4,annual crop land,annual crop land,9,8,base,5,4,4,4,Explanation A provides a broader set of concre...


In [81]:
win_counts = df_exp12["winner"].value_counts()

print(win_counts)

winner
base       56
GradCAM    42
Tie        22
Name: count, dtype: int64


In [82]:
total = len(df_exp12)

gradcam_wins = (df_exp12["winner"] == "GradCAM").sum()
base_wins = (df_exp12["winner"] == "base").sum()
ties = (df_exp12["winner"] == "Tie").sum()

print(f"GradCAM win rate: {gradcam_wins/total*100:.2f}%")
print(f"Baseline win rate: {base_wins/total*100:.2f}%")
print(f"Tie rate: {ties/total*100:.2f}%")

GradCAM win rate: 35.00%
Baseline win rate: 46.67%
Tie rate: 18.33%


In [83]:
avg_base = df_exp12["score_base"].mean()
avg_gradcam = df_exp12["score_gradcam"].mean()

print(f"Avg Base Score: {avg_base:.2f}")
print(f"Avg GradCAM Score: {avg_gradcam:.2f}")


Avg Base Score: 8.15
Avg GradCAM Score: 8.21


In [84]:
improvement = ((avg_gradcam - avg_base) / avg_base) * 100

print(f"Improvement: {improvement:.2f}%")

Improvement: 0.72%


# Exp1 vs Exp3 (base vs lime)

In [85]:
results_base_vs_lime = []

for i in range(len(exp1)):

    predicted_class = exp1.loc[i, "predicted_class"]

    exp1_text = exp1.loc[i, "direct_description"]
    exp3_text = exp3.loc[i, "lime_explanation"]

    prompt = build_judge_prompt(
        predicted_class,
        exp1_text,
        exp3_text,
        "LIME"
    )

    result = judge_with_openai(prompt)

    # Map winner labels
    winner_raw = result.get("winner")

    if winner_raw == "A":
        winner_label = "base"
    elif winner_raw == "B":
        winner_label = "LIME"
    elif winner_raw == "Tie":
        winner_label = "Tie"
    else:
        winner_label = "Error"

    results_base_vs_lime.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "score_base": result.get("score_A"),
        "score_lime": result.get("score_B"),
        "winner": winner_label,
        "specificity_base": result.get("specificity_A"),
        "specificity_lime": result.get("specificity_B"),
        "usefulness_base": result.get("usefulness_A"),
        "usefulness_lime": result.get("usefulness_B"),
        "reason": result.get("reason")
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [86]:
df_exp13 = pd.DataFrame(results_base_vs_lime)

df_exp13.head()

,index,true_class,predicted_class,score_base,score_lime,winner,specificity_base,specificity_lime,usefulness_base,usefulness_lime,reason
0,0,annual crop land,annual crop land,9,9,Tie,5,5,4,4,Both explanations provide multiple specific vi...
1,1,annual crop land,annual crop land,8,9,LIME,4,5,4,4,Explanation B provides more precise spatial de...
2,2,annual crop land,annual crop land,9,8,base,5,4,4,4,Explanation A provides more multiple specific ...
3,3,annual crop land,annual crop land,8,9,LIME,4,5,4,4,Explanation B provides more specific visual de...
4,4,annual crop land,annual crop land,8,9,LIME,4,5,4,4,Explanation B provides a more detailed and spe...


In [87]:
win_counts = df_exp13["winner"].value_counts()
print(win_counts)

winner
LIME    65
base    47
Tie      8
Name: count, dtype: int64


In [88]:
total = len(df_exp13)

lime_wins = (df_exp13["winner"] == "LIME").sum()
base_wins = (df_exp13["winner"] == "base").sum()
ties = (df_exp13["winner"] == "Tie").sum()

print(f"LIME win rate: {lime_wins/total*100:.2f}%")
print(f"Baseline win rate: {base_wins/total*100:.2f}%")
print(f"Tie rate: {ties/total*100:.2f}%")

LIME win rate: 54.17%
Baseline win rate: 39.17%
Tie rate: 6.67%


In [89]:
avg_base = df_exp13["score_base"].mean()
avg_lime = df_exp13["score_lime"].mean()

print(f"Avg Base Score: {avg_base:.2f}")
print(f"Avg LIME Score: {avg_lime:.2f}")

Avg Base Score: 7.92
Avg LIME Score: 8.37


In [90]:
improvement = ((avg_lime - avg_base) / avg_base) * 100

print(f"Improvement: {improvement:.2f}%")

Improvement: 5.68%


# Exp1 vs Exp4 (base vs shap)

In [91]:
results_base_vs_shap = []

for i in range(len(exp1)):

    predicted_class = exp1.loc[i, "predicted_class"]

    exp1_text = exp1.loc[i, "direct_description"]
    exp4_text = exp4.loc[i, "shap_explanation"]

    prompt = build_judge_prompt(
        predicted_class,
        exp1_text,
        exp4_text,
        "SHAP"
    )

    result = judge_with_openai(prompt)

    # Map winner labels
    winner_raw = result.get("winner")

    if winner_raw == "A":
        winner_label = "base"
    elif winner_raw == "B":
        winner_label = "SHAP"
    elif winner_raw == "Tie":
        winner_label = "Tie"
    else:
        winner_label = "Error"

    results_base_vs_shap.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "score_base": result.get("score_A"),
        "score_shap": result.get("score_B"),
        "winner": winner_label,
        "specificity_base": result.get("specificity_A"),
        "specificity_shap": result.get("specificity_B"),
        "usefulness_base": result.get("usefulness_A"),
        "usefulness_shap": result.get("usefulness_B"),
        "reason": result.get("reason")
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [92]:
df_exp14 = pd.DataFrame(results_base_vs_shap)

df_exp14.head()

,index,true_class,predicted_class,score_base,score_shap,winner,specificity_base,specificity_shap,usefulness_base,usefulness_shap,reason
0,0,annual crop land,annual crop land,9,9,Tie,5,5,4,4,Both explanations provide multiple specific vi...
1,1,annual crop land,annual crop land,8,9,SHAP,4,5,4,4,Explanation B provides more precise spatial de...
2,2,annual crop land,annual crop land,9,7,base,5,3,4,4,Explanation A provides more detailed and concr...
3,3,annual crop land,annual crop land,9,8,base,5,4,4,4,Explanation A provides more detailed and varie...
4,4,annual crop land,annual crop land,9,9,Tie,5,5,4,4,Both explanations provide multiple specific vi...


In [93]:
win_counts = df_exp14["winner"].value_counts()
print(win_counts)

winner
base    62
SHAP    43
Tie     15
Name: count, dtype: int64


In [94]:
total = len(df_exp14)

shap_wins = (df_exp14["winner"] == "SHAP").sum()
base_wins = (df_exp14["winner"] == "base").sum()
ties = (df_exp14["winner"] == "Tie").sum()

print(f"SHAP win rate: {shap_wins/total*100:.2f}%")
print(f"Baseline win rate: {base_wins/total*100:.2f}%")
print(f"Tie rate: {ties/total*100:.2f}%")

SHAP win rate: 35.83%
Baseline win rate: 51.67%
Tie rate: 12.50%


In [95]:
avg_base = df_exp14["score_base"].mean()
avg_shap = df_exp14["score_shap"].mean()

print(f"Avg Base Score: {avg_base:.2f}")
print(f"Avg SHAP Score: {avg_shap:.2f}")

Avg Base Score: 8.21
Avg SHAP Score: 8.10


In [96]:
improvement = ((avg_shap - avg_base) / avg_base) * 100

print(f"Improvement: {improvement:.2f}%")

Improvement: -1.32%


# Save SHAP results to CSV

In [98]:
df_exp12.to_csv(f"exp2_gradcam_evaluation.csv", index=False)

In [99]:
df_exp13.to_csv(f"exp3_lime_evaluation.csv", index=False)

In [100]:
df_exp14.to_csv(f"exp4_shap_evaluation.csv", index=False)

# Exp5

In [112]:
def build_combined_judge_prompt(predicted_class, baseline_explanation, combined_explanation):
    
    return f"""You are evaluating two explanations for why a CNN model predicted a specific class for a satellite image.

The model predicted: "{predicted_class}"

Explanation A :
\"\"\"{baseline_explanation}\"\"\"

Explanation B :
\"\"\"{combined_explanation}\"\"\"

Evaluate both explanations on these criteria:

1. Specificity (0-5 points):
   - Does the explanation mention concrete visual features?
   - Examples: shapes (rectangular, circular), textures (rough, smooth), 
     patterns (grid-like, uniform), colors (green, brown), 
     spatial layout (clustered, linear, scattered)
   - Avoid vague terms like "typical" or "characteristic"
   
   Score 5: Multiple specific visual features clearly described
   Score 3: Some specific features mentioned
   Score 1: Only vague, general descriptions
   Score 0: No visual features mentioned

2. Usefulness (0-5 points):
   - Does the explanation help a human understand the model's reasoning?
   - Does it connect visual features to the predicted class?
   - Would a non-technical user (city planner, policy maker) find this explanation helpful?
   
   Score 5: Very clear connection between features and prediction
   Score 3: Some helpful information but not fully clear
   Score 1: Minimal help in understanding the reasoning
   Score 0: Confusing or unhelpful

Total Score = Specificity + Usefulness (0 to 10)

Return ONLY valid JSON. Do NOT use markdown formatting or code blocks.

{{
  "score_A": <number 0-10>,
  "score_B": <number 0-10>,
  "specificity_A": <number 0-5>,
  "specificity_B": <number 0-5>,
  "usefulness_A": <number 0-5>,
  "usefulness_B": <number 0-5>,
  "winner": "A" or "B" or "Tie",
  "reason": "<brief 1-sentence explanation of why winner was chosen>"
}}
"""

In [111]:
df_exp5 = pd.DataFrame({
    "index": exp1.index,
    "true_class": exp1["true_class"],
    "predicted_class": exp1["predicted_class"],
    
    # Combine explanations from 3 XAI methods
    "combined_explanation": (
        exp2["xai_explanation"] + " " +
        exp3["lime_explanation"] + " " +
        exp4["shap_explanation"]
    )
})

df_exp5.head()

,index,true_class,predicted_class,combined_explanation
0,0,annual crop land,annual crop land,"The model likely predicted ""annual crop land"" ..."
1,1,annual crop land,annual crop land,"The model likely predicted ""annual crop land"" ..."
2,2,annual crop land,annual crop land,"The model likely predicted ""annual crop land"" ..."
3,3,annual crop land,annual crop land,"The model likely predicted ""annual crop land"" ..."
4,4,annual crop land,annual crop land,"The model likely predicted ""annual crop land"" ..."


In [110]:
df_exp5.head()

,index,true_class,predicted_class,combined_explanation
0,0,annual crop land,annual crop land,"GradCAM: The model likely predicted ""annual cr..."
1,1,annual crop land,annual crop land,"GradCAM: The model likely predicted ""annual cr..."
2,2,annual crop land,annual crop land,"GradCAM: The model likely predicted ""annual cr..."
3,3,annual crop land,annual crop land,"GradCAM: The model likely predicted ""annual cr..."
4,4,annual crop land,annual crop land,"GradCAM: The model likely predicted ""annual cr..."


In [119]:
results_base_vs_combined = []

for i in range(len(exp1)):

    predicted_class = exp1.loc[i, "predicted_class"]

    # Baseline explanation
    exp1_text = exp1.loc[i, "direct_description"]

    # Combined explanation (Exp5 output)
    exp5_text = df_exp5.loc[i, "combined_explanation"]

    # 🔥 Build prompt
    prompt = build_combined_judge_prompt(
        predicted_class,
        exp1_text,
        exp5_text
    )

    # 🔥 Call judge (OpenAI)
    result = judge_with_openai(prompt)

    # Map winner labels
    winner_raw = result.get("winner")

    if winner_raw == "A":
        winner_label = "base"
    elif winner_raw == "B":
        winner_label = "Combined"
    elif winner_raw == "Tie":
        winner_label = "Tie"
    else:
        winner_label = "Error"

    # Save results
    results_base_vs_combined.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "score_base": result.get("score_A"),
        "score_combined": result.get("score_B"),
        "specificity_base": result.get("specificity_A"),
        "specificity_combined": result.get("specificity_B"),
        "usefulness_base": result.get("usefulness_A"),
        "usefulness_combined": result.get("usefulness_B"),
        "winner": winner_label,
        "reason": result.get("reason")
    })

    # Progress update
    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [ ]:
# Pick first sample
i = 6

predicted_class = exp1.loc[i, "predicted_class"]

# Exp1 = baseline
exp1_text = exp1.loc[i, "direct_description"]
print("exp1_text:")
print(exp1_text)

# Exp2 = GradCAM (you can later switch to exp3 or exp4)
exp2_text = df_exp5.loc[i, "combined_explanation"]
print("\nexp2_text:")
print(exp2_text)

# Build prompt (no bias mention)
prompt = build_combined_judge_prompt(
    predicted_class,
    exp1_text,
    exp2_text,
)

# Call openai
result = judge_with_openai(prompt)

print("\nResult:")
print(result)

exp1_text:
The image shows distinct geometric shapes and patterns typical of agricultural fields, with clear edges and uniform, segmented areas. The varying colors and tones suggest different stages of crop growth or different types of crops being cultivated. The regularity and structured layout are characteristic of land used for annual crop cultivation, where plots are systematically organized for planting and harvesting cycles.

exp2_text:
The model likely predicted "annual crop land" because the bottom-left region reveals a distinct patchwork pattern, characteristic of agricultural fields. The green area suggests active vegetation, typical of crops in growth stages. Regular, geometric shapes are evident, indicating human cultivation, a common trait in farmed land. The model likely predicted "annual crop land" due to the presence of geometric, patchwork patterns in the lower-left, lower-center, and upper-right segments. These areas show distinct, organized rectangles and trapezoids 

In [120]:
df_exp15 = pd.DataFrame(results_base_vs_combined)

In [121]:
print(df_exp15["winner"].value_counts())

winner
Combined    117
base          3
Name: count, dtype: int64


In [122]:
total = len(df_exp15)

combined_wins = (df_exp15["winner"] == "Combined").sum()
base_wins = (df_exp15["winner"] == "base").sum()
ties = (df_exp15["winner"] == "Tie").sum()

print(f"Combined win rate: {combined_wins/total*100:.2f}%")
print(f"Baseline win rate: {base_wins/total*100:.2f}%")
print(f"Tie rate: {ties/total*100:.2f}%")

Combined win rate: 97.50%
Baseline win rate: 2.50%
Tie rate: 0.00%


In [123]:
avg_base = df_exp15["score_base"].mean()
avg_combined = df_exp15["score_combined"].mean()

print(f"Avg Base Score: {avg_base:.2f}")
print(f"Avg Combined Score: {avg_combined:.2f}")

Avg Base Score: 6.86
Avg Combined Score: 9.26


In [124]:
improvement = ((avg_combined - avg_base) / avg_base) * 100
print(f"Improvement: {improvement:.2f}%")

Improvement: 34.99%


In [125]:
df_exp15.to_csv(f"exp5_base_vs_combined.csv", index=False)

# EXP 6

In [126]:
def compress_combined_explanation(predicted_class, gradcam_text, lime_text, shap_text):

    prompt = f"""You are given three short explanations of why a CNN predicted a class for a satellite image.

Predicted class: "{predicted_class}"

Explanation 1:
{gradcam_text}

Explanation 2:
{lime_text}

Explanation 3:
{shap_text}

Your task:
- Combine these into ONE concise explanation
- Remove repeated information
- Do not add any new explanation
- Keep length similar to a normal explanation (3–4 sentences)

Return only the final combined explanation (no extra text)."""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_completion_tokens=200
    )

    return response.choices[0].message.content.strip()

In [127]:
exp5_clean = []

for i in range(len(exp1)):

    predicted_class = exp1.loc[i, "predicted_class"]

    gradcam_text = exp2.loc[i, "xai_explanation"]
    lime_text = exp3.loc[i, "lime_explanation"]
    shap_text = exp4.loc[i, "shap_explanation"]

    combined_clean = compress_combined_explanation(
        predicted_class,
        gradcam_text,
        lime_text,
        shap_text
    )

    exp5_clean.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "combined_explanation": combined_clean
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

df_exp5_clean = pd.DataFrame(exp5_clean)

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [ ]:
# Pick first sample
i = 5
predicted_class = exp1.loc[i, "predicted_class"]

# Exp1 = baseline
exp1_text = exp1.loc[i, "direct_description"]
print("exp1_text:")
print(exp1_text)

# Exp2 = GradCAM (you can later switch to exp3 or exp4)
exp2_text = df_exp5_clean.loc[i, "combined_explanation"]
print("\nexp2_text:")
print(exp2_text)

# Build prompt (no bias mention)
prompt = build_combined_judge_prompt(
    predicted_class,
    exp1_text,
    exp2_text,
)

# Call openai
result = judge_with_openai(prompt)

print("\nResult:")
print(result)

exp1_text:
The model likely predicted "annual crop land" due to the presence of rectangular or geometric patterns, which are indicative of farmland plots. The texture appears consistent with that of cultivated soil, with variations in color or shading suggesting different stages of crop growth or land preparation. The organized appearance and defined boundaries are typical characteristics of agricultural areas managed for annual crops.

exp2_text:
The model likely predicted "annual crop land" because the image displays distinct rectangular plots with uniform textures and clear, straight boundaries, characteristic of cultivated agricultural fields. These rectilinear patterns, visible in the upper-left, upper-center, and upper-right regions, suggest human planning and managed cropping systems. The arrangement and uniformity of these areas contrast with the irregular patterns typical of natural or non-agricultural land.

Result:
{'score_A': 7, 'score_B': 9, 'specificity_A': 3, 'specificit

In [132]:
results_base_vs_combined2 = []

for i in range(len(exp1)):

    predicted_class = exp1.loc[i, "predicted_class"]

    # Baseline explanation
    exp1_text = exp1.loc[i, "direct_description"]

    # Combined explanation (Exp5 output)
    exp5_text = df_exp5_clean.loc[i, "combined_explanation"]

    # 🔥 Build prompt
    prompt = build_combined_judge_prompt(
        predicted_class,
        exp1_text,
        exp5_text
    )

    # 🔥 Call judge (OpenAI)
    result = judge_with_openai(prompt)

    # Map winner labels
    winner_raw = result.get("winner")

    if winner_raw == "A":
        winner_label = "base"
    elif winner_raw == "B":
        winner_label = "Combined"
    elif winner_raw == "Tie":
        winner_label = "Tie"
    else:
        winner_label = "Error"

    # Save results
    results_base_vs_combined2.append({
        "index": i,
        "true_class": exp1.loc[i, "true_class"],
        "predicted_class": predicted_class,
        "score_base": result.get("score_A"),
        "score_combined": result.get("score_B"),
        "specificity_base": result.get("specificity_A"),
        "specificity_combined": result.get("specificity_B"),
        "usefulness_base": result.get("usefulness_A"),
        "usefulness_combined": result.get("usefulness_B"),
        "winner": winner_label,
        "reason": result.get("reason")
    })

    # Progress update
    if i % 10 == 0:
        print(f"Processed {i}/{len(exp1)}")

Processed 0/120
Processed 10/120
Processed 20/120
Processed 30/120
Processed 40/120
Processed 50/120
Processed 60/120
Processed 70/120
Processed 80/120
Processed 90/120
Processed 100/120
Processed 110/120


In [133]:
df_exp16 = pd.DataFrame(results_base_vs_combined2)

In [134]:
print(df_exp16["winner"].value_counts())

winner
Combined    91
Tie         21
base         8
Name: count, dtype: int64


In [136]:

total = len(df_exp16)

combined_wins = (df_exp16["winner"] == "Combined").sum()
base_wins = (df_exp16["winner"] == "base").sum()
ties = (df_exp16["winner"] == "Tie").sum()

print(f"Combined win rate: {combined_wins/total*100:.2f}%")
print(f"Baseline win rate: {base_wins/total*100:.2f}%")
print(f"Tie rate: {ties/total*100:.2f}%")

avg_base = df_exp16["score_base"].mean()
avg_combined = df_exp16["score_combined"].mean()

print(f"Avg Base Score: {avg_base:.2f}")
print(f"Avg Combined Score: {avg_combined:.2f}")

improvement = ((avg_combined - avg_base) / avg_base) * 100
print(f"Improvement: {improvement:.2f}%")


Combined win rate: 75.83%
Baseline win rate: 6.67%
Tie rate: 17.50%
Avg Base Score: 7.57
Avg Combined Score: 8.97
Improvement: 18.50%


In [137]:
df_exp16.to_csv(f"exp6_base_vs_combined_clean.csv", index=False)